# *TASK 1: End-to-End ML System with Streamlit GUI*

<p style='font-size:20px;'><i>Customer churn is a critical issue for businesses, as losing customers directly impacts revenue. The objective of this task is to build a complete machine learning pipeline that predicts whether a customer will churn and deploy it using an interactive Streamlit GUI for real-time predictions.</i></p>

In [15]:
# Importing all the required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

In [16]:
# Load the customer churn dataset into pandas dataframe
df = pd.read_csv('Churn_Modelling.csv')

In [17]:
# Display the first five rows of the dataframe to get a quick preview 
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [18]:
# Get the number of rows and columns of the dataframe
df.shape

(10000, 14)

In [19]:
# Display concise overview of the dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [20]:
# Check for any null values in the dataframe
df.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [21]:
# Check for any duplicate rows in the dataframe
df.duplicated().sum()

0

In [22]:
# Drop the unnecesssary columns from the dataframe
df.drop(['RowNumber','CustomerId','Surname'], axis=1, inplace=True)

In [23]:
# Import OneHotEncoder from sklearn's preprocessing module for nominal categorical data encoding
from sklearn.preprocessing import OneHotEncoder
# Create an encoder instance
one_hot_encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore',dtype=np.int32)

In [24]:
# Fit and transform the categorical features
df_encoded = one_hot_encoder.fit_transform(df[['Geography','Gender']])

In [25]:
# Convert to a dataframe
df_encoded = pd.DataFrame(df_encoded, columns=one_hot_encoder.get_feature_names_out(['Geography', 'Gender']))
# Concatenate with original dataframe (excluding encoded columns)
df_encoded = pd.concat([df.drop(['Geography', 'Gender'], axis=1), df_encoded], axis=1)

In [26]:
# Display the final dataframe after one hot encoding
df_encoded

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,0,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,1,0
2,502,42,8,159660.80,3,1,0,113931.57,1,0,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,39,5,0.00,2,1,0,96270.64,0,0,0,1
9996,516,35,10,57369.61,1,1,1,101699.77,0,0,0,1
9997,709,36,7,0.00,1,0,1,42085.58,1,0,0,0
9998,772,42,3,75075.31,2,1,0,92888.52,1,1,0,1


In [54]:
# Separate features (x) and target variable (y)
x = df_encoded.drop('Exited', axis=1)
y = df_encoded['Exited']
# Split the dataset into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [62]:
# Import SMOTE (Synthetic Minority Over-sampling Technique) from imbalanced-learn library
from imblearn.over_sampling import SMOTE
# Initialize SMOTE
smote = SMOTE(random_state=42)
# Apply SMOTE to balance the training dataset
x_train_bal, y_train_bal = smote.fit_resample(x_train, y_train)

In [63]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(x_train_bal, y_train_bal)

E:\Anaconda\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [64]:
# Import RandomForest from sklearns's ensemble module to build and train a classification model
from sklearn.ensemble import RandomForestClassifier

In [65]:
# Instantiate and fit the model
rf = RandomForestClassifier()
rf.fit(x_train_bal, y_train_bal)

RandomForestClassifier()

In [66]:
# Import xgboost to build and train a classification model
from xgboost import XGBClassifier
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(x_train_bal, y_train_bal)

E:\Anaconda\Lib\site-packages\xgboost\training.py:183: UserWarning: [21:05:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [67]:
# Predictions
lr_pred = lr.predict(x_test)
rf_pred = rf.predict(x_test)
xgb_pred = xgb.predict(x_test)

# Accuracy
lr_acc = accuracy_score(y_test, lr_pred)
rf_acc = accuracy_score(y_test, rf_pred)
xgb_acc = accuracy_score(y_test, xgb_pred)

print("Logistic Regression:", lr_acc)
print("Random Forest:", rf_acc)
print("XGBoost:", xgb_acc)

Logistic Regression: 0.6505
Random Forest: 0.817
XGBoost: 0.8145


In [68]:
models = {
    "Logistic Regression": lr_acc,
    "Random Forest": rf_acc,
    "XGBoost": xgb_acc
}

best_model_name = max(models, key=models.get)
print("Best Model:", best_model_name)

if best_model_name == "Logistic Regression":
    best_model = lr
elif best_model_name == "Random Forest":
    best_model = rf
else:
    best_model = xgb

Best Model: Random Forest


In [69]:
# Saving the best model
import joblib

joblib.dump(best_model, "best_model.pkl")
joblib.dump(x.columns.tolist(), "features.pkl")
joblib.dump(xgb_acc, "accuracy.pkl")

['accuracy.pkl']

In [ ]:
!streamlit run app.py

# *Conclusion*

<p style ='font-size:20px';><i>This task demonstrated the complete lifecycle of a machine learning system, from data preprocessing to deployment. The integration with Streamlit made the solution user-friendly and practical for real-world applications.</i></p>